In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.sparse as sp
from scipy.sparse.linalg import spsolve
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

from src.particle_simulation import get_acceleration, simulate

N = 4000     # Number of particles
Nx = 400      # Number of mesh cells
tEnd = 50     # time at which simulation ends
dt = 1        # timestep
boxsize = 50  # periodic domain [0,boxsize]
n0 = 1        # electron number density
vb = 3        # beam velocity
vth = 1       # beam width
A = 0.1       # perturbation

# Generate Initial Conditions
np.random.seed(42)
pos = np.random.rand(N, 1) * boxsize
vel = vth * np.random.randn(N, 1) + vb
Nh = int(N / 2)
vel[Nh:] *= -1
vel *= 1 + A * np.sin(2 * np.pi * pos / boxsize)

# Construct matrix G to compute Gradient (1st derivative)
dx = boxsize / Nx
e = np.ones(Nx)
diags = np.array([-1, 1])
vals = np.vstack((-e, e))
Gmtx = sp.spdiags(vals, diags, Nx, Nx).tolil()
Gmtx[0, Nx - 1] = -1
Gmtx[Nx - 1, 0] = 1
Gmtx /= 2 * dx
Gmtx = Gmtx.tocsr()

# Construct matrix L to compute Laplacian (2nd derivative)
diags = np.array([-1, 0, 1])
vals = np.vstack((e, -2 * e, e))
Lmtx = sp.spdiags(vals, diags, Nx, Nx).tolil()
Lmtx[0, Nx - 1] = 1
Lmtx[Nx - 1, 0] = 1
Lmtx /= dx**2
Lmtx = Lmtx.tocsr()

# Calculate initial accelerations
acc = get_acceleration(pos, vel, Nx, boxsize, n0, Gmtx, Lmtx)


fig, ax = plt.subplots(figsize=(6, 4), dpi=80)

# Plot initial state and keep references to the scatter objects

pos, vel, acc, scatter1, scatter2 = simulate(pos, A, Nh, fig, ax, dt, vel, acc, tEnd, Nx, boxsize, n0, Gmtx, Lmtx)

Nt = int(np.ceil(tEnd / dt))

def update_frame(frame):
    """
    Update the animation frame
    """
    global pos, vel, acc
    pos, vel, acc, scatter1, scatter2 = simulate(pos, A, Nh, fig, ax, dt, vel, acc, tEnd, Nx, boxsize, n0, Gmtx, Lmtx)
    return scatter1, scatter2

ani = FuncAnimation(
    fig,
    update_frame,
    frames=Nt,
    interval=50, # Milliseconds between frames
    blit=True
)

# To display in a Jupyter Notebook
HTML(ani.to_jshtml())